In [1]:
import pandas as pd
import os

if os.path.exists('/root/Public_Storage/madelab_khw/lg_aimers/dataset/train.csv'):
    df = pd.read_csv('/root/Public_Storage/madelab_khw/lg_aimers/dataset/train.csv')
else:
    print('none')

os.system('nvidia-smi')

Tue Feb 25 09:36:06 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.183.01             Driver Version: 535.183.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA RTX A5000               Off | 00000000:01:00.0 Off |                  Off |
| 30%   28C    P8              19W / 230W |      9MiB / 24564MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

0

In [2]:
import sys
print(sys.executable)


/root/anaconda3/envs/khw/bin/python


In [3]:
import torch
if torch.cuda.is_available() :
    # torch.cuda.device # Context-manager that changes the selected device.
    device = torch.device('cuda:0')
    device_count = torch.cuda.device_count()
    print('GPU is available.')
else :
    device = torch.device('cpu')
    print('GPU is not available.')

print(device)

GPU is available.
cuda:0


In [4]:


device = torch.device("cuda:0" if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [5]:
def seed_everything(seed = 21):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

In [6]:
print(df.shape)

(256351, 69)


In [7]:
def convert(str):
    return f'{str}'

def pre_process(df):
    threshold = 0.8
    df = df.loc[:, df.isnull().mean()<0.8]
    df = df.drop(columns=['ID'])
    df_sol = df['임신 성공 여부']
    df = df.drop(columns=['임신 성공 여부'])
    
    with open('/root/Public_Storage/madelab_khw/lg_aimers/code/import.txt', 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    lines = list(map(str.strip, lines))
    lines = list(map(convert, lines))
    
    print(lines)
    
    for col in df.columns:
        if col not in lines:
            df = df.drop(columns=[col])
    
    print(df.columns.tolist)
    
    
    for col in df.columns:
        if df[col].dtype == 'object':  # 문자열 컬럼 처리
            df[col].fillna('Unknown', inplace=True)
        else:
            unique_count = df[col].nunique()
            if unique_count <= 4:
                df[col].fillna(0, inplace=True)
            else:
                df[col].fillna(df[col].mean(), inplace=True)
    
    
    return df, df_sol
        

In [8]:
pre_df, sol = pre_process(df)
print(sol.shape)

['이식된 배아 수', '시술 유형', '배아 이식 경과일', '저장된 배아 수', '동결 배아 사용 여부', '배아 생성 주요 이유', '신선 배아 사용 여부', '난자 출처', '시술 당시 나이', '총 생성 배아 수', '단일 배아 이식 여부', '미세주입 배아 이식 수', '난자 기증자 나이', '해동 난자 수', '배란 유도 유형', '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수', '특정 시술 유형', '정자 출처', '미세주입 후 저장된 배아 수', 'IVF 임신 횟수', '해동된 배아 수', 'DI 출산 횟수', '총 출산 횟수', 'IVF 출산 횟수', '불임 원인 - 난관 질환', '파트너 정자와 혼합된 난자 수', '총 임신 횟수', '미세주입에서 생성된 배아 수']
<bound method IndexOpsMixin.tolist of Index(['시술 당시 나이', '시술 유형', '특정 시술 유형', '배란 유도 유형', '단일 배아 이식 여부',
       '불임 원인 - 난관 질환', '배아 생성 주요 이유', '총 임신 횟수', 'IVF 임신 횟수', '총 출산 횟수',
       'IVF 출산 횟수', 'DI 출산 횟수', '총 생성 배아 수', '미세주입에서 생성된 배아 수', '이식된 배아 수',
       '미세주입 배아 이식 수', '저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수', '해동 난자 수',
       '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수', '파트너 정자와 혼합된 난자 수', '난자 출처',
       '정자 출처', '난자 기증자 나이', '동결 배아 사용 여부', '신선 배아 사용 여부', '배아 이식 경과일'],
      dtype='object')>
(256351,)


In [9]:
import pandas as pd

pd.set_option('display.max_rows', None)  # 모든 행 출력
pd.set_option('display.max_columns', None)  # 모든 열 출력
pd.set_option('display.expand_frame_repr', False)  # 가로 생략 방지


In [10]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

def pre_tf(pre_df, sol):
    category_columns = pre_df.select_dtypes(include=['object']).columns.tolist()
    numeric_columns = pre_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
        
    label_encoder = {}
    for col in category_columns:
        encoder = LabelEncoder()
        pre_df[col] = encoder.fit_transform(pre_df[col])
    
    X = pre_df[category_columns + numeric_columns]  
    y = sol  
    
    X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=pre_df['시술 당시 나이'])
    return X_train, X_valid, y_train, y_valid

In [11]:
X_train, X_valid, y_train, y_valid = pre_tf(pre_df, sol)

category_columns = pre_df.select_dtypes(include=['object']).columns.tolist()
numeric_columns = pre_df.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [12]:
os.system('pip install imbalanced-learn')


[notice] A new release of pip available: 22.3.1 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


0

In [13]:
from imblearn.over_sampling import SMOTE

# ✅ SMOTE 적용하여 클래스 균형 맞추기
sm = SMOTE(random_state=42)
X_train, y_train = sm.fit_resample(X_train, y_train)

In [14]:
import pandas as pd
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
import numpy as np

X_train_numpy = X_train.to_numpy().astype(np.float32)
X_valid_numpy = X_valid.to_numpy().astype(np.float32)
y_train_numpy = y_train.to_numpy().astype(np.int64)
y_valid_numpy = y_valid.to_numpy().astype(np.int64)

print(X_train_numpy.shape)


(304470, 30)


In [15]:


# ✅ TabNet 모델 생성 (scale_pos_weight 제거)
model = TabNetClassifier(
    cat_idxs=[X_train.columns.get_loc(col) for col in category_columns],
    cat_dims=[len(X_train[col].unique()) for col in category_columns],
    cat_emb_dim=8,  
    optimizer_fn=torch.optim.Adam,
    optimizer_params={'lr': 1e-2},
    scheduler_params={"step_size": 10, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax'  
)

# ✅ 가중치 적용하여 모델 학습

model.fit(
    X_train=X_train_numpy, y_train=y_train_numpy,
    eval_set=[(X_valid_numpy, y_valid_numpy)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    max_epochs=100,
    patience=10,
    batch_size=256,
    virtual_batch_size=64,
    num_workers=0,
    drop_last=False,
    
)

y_pred = model.predict(X_valid_numpy)

acc = accuracy_score(y_valid_numpy, y_pred)
f1 = f1_score(y_valid_numpy, y_pred, average='macro')  
recall = recall_score(y_valid_numpy, y_pred, average='macro')
precision = precision_score(y_valid_numpy, y_pred, average='macro')

print('-------------result------------')
print(f" Accuracy  : {acc:.4f}")
print(f" F1 Score  : {f1:.4f}")
print(f" Recall    : {recall:.4f}")
print(f" Precision : {precision:.4f}")


/root/anaconda3/envs/khw/lib/python3.8/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

import xgboost as xgb

# 데이터 DMatrix로 변환
X_train_aligned = pd.DataFrame(X_train_numpy, columns=X_train.columns).astype(np.float32)
X_valid_aligned = pd.DataFrame(X_valid_numpy, columns=X_train.columns).astype(np.float32)

dtrain = xgb.DMatrix(X_train_aligned, label=y_train_numpy, feature_names=X_train.columns.tolist())
dvalid = xgb.DMatrix(X_valid_aligned, label=y_valid_numpy, feature_names=X_train.columns.tolist())

# XGBoost 파라미터 설정
params = {
    "objective": "binary:logistic",
    "learning_rate": 0.01,
    "max_depth": 6,
    "colsample_bytree": 0.8,
    "subsample": 0.8,
    "eval_metric": "logloss"}



params["objective"] = "binary:logistic"
xgb_model = xgb.train(params, dtrain, num_boost_round=500, 
                      evals=[(dvalid, "validation")],early_stopping_rounds=50)




xgb_model.feature_names = X_train.columns.tolist()

y_pred_proba_xgb = xgb_model.predict(dvalid)  # XGB 확률값 (0~1)
y_pred_xgb = (y_pred_proba_xgb >= 0.5).astype(int)  # 🚀 확률 → 0 또는 1 변환

# ✅ TabNet 예측
y_pred_proba_tabnet = model.predict_proba(X_valid.to_numpy().astype(np.float32))[:, 1]

# ✅ XGB + TabNet Soft Voting
y_pred_proba_ensemble = (y_pred_proba_xgb + y_pred_proba_tabnet) / 2
y_pred_ensemble = (y_pred_proba_ensemble >= 0.5).astype(int)  # 🚀 확률 → 0 또는 1 변환

# 📌 (1) XGB 모델 평가
acc_xgb = accuracy_score(y_valid, y_pred_xgb)
f1_xgb = f1_score(y_valid, y_pred_xgb, average='macro')  
recall_xgb = recall_score(y_valid, y_pred_xgb, average='macro')
precision_xgb = precision_score(y_valid, y_pred_xgb, average='macro')
roc_auc_xgb = roc_auc_score(y_valid, y_pred_proba_xgb)  # ✅ 확률값을 사용해야 함

# 📌 (2) 앙상블 모델 평가
acc_ensemble = accuracy_score(y_valid, y_pred_ensemble)
f1_ensemble = f1_score(y_valid, y_pred_ensemble, average='macro')  
recall_ensemble = recall_score(y_valid, y_pred_ensemble, average='macro')
precision_ensemble = precision_score(y_valid, y_pred_ensemble, average='macro')
roc_auc_ensemble = roc_auc_score(y_valid, y_pred_proba_ensemble)  # ✅ 확률값 사용

# 📌 결과 출력
print('-------------XGBoost------------')
print(f" Accuracy  : {acc_xgb:.4f}")
print(f" F1 Score  : {f1_xgb:.4f}")
print(f" Recall    : {recall_xgb:.4f}")
print(f" Precision : {precision_xgb:.4f}")
print(f" 🚀 ROC-AUC Score : {roc_auc_xgb:.4f}")

print('-------------Ensemble------------')
print(f" Accuracy  : {acc_ensemble:.4f}")
print(f" F1 Score  : {f1_ensemble:.4f}")
print(f" Recall    : {recall_ensemble:.4f}")
print(f" Precision : {precision_ensemble:.4f}")
print(f" 🚀 ROC-AUC Score : {roc_auc_ensemble:.4f}")

In [ ]:
# xgb_model.save_model("new_xgb_model.model")
# model.save_model("new_tabnet.zip")
                 

In [ ]:
# print(xgb_model.feature_names)

In [ ]:
# print(type(feature_importance[0][0]))

In [ ]:
# feature_importance = xgb_model.get_score(importance_type='gain')  # 중요도 측정
# feature_importance = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True) 

In [ ]:
# num = 0
# with open('/root/Public_Storage/madelab_khw/lg_aimers/code/import.txt', 'w', encoding = 'utf-8') as f:
#     for res in feature_importance:
#         if num == 30:
#             break
#         f.write(str(res[0])+'\n')
#         num+=1
    
    
        

In [ ]:
import matplotlib.pyplot as plt

plt.hist(y_pred_proba_xgb, bins=50, alpha=0.5, label="XGBoost")
plt.hist(y_pred_proba_tabnet, bins=50, alpha=0.5, label="TabNet")
plt.legend()
plt.title("XGB vs. TabNet 확률 분포")
plt.show()
